# 🕹️ Deep RL Evolution: REINFORCE → A2C → A2C+GAE → PPO

**Environment:** `ALE/Pong-v5` from raw pixels — 4-frame grayscale stack, 84×84  
**Runtime:** GPU (T4 recommended) · Runtime → Change runtime type → GPU

---

## Section 1 · Setup & Virtual Display

In [ ]:
# System packages
!apt-get update -qq
!apt-get install -y -qq xvfb ffmpeg

# Python packages
!pip install -q \
    "setuptools<82" \
    jedi \
    "gymnasium[atari]" \
    "autorom[accept-rom-license]" \
    opencv-python-headless \
    pyvirtualdisplay \
    matplotlib \
    pandas \
    imageio \
    imageio-ffmpeg

# Install Atari ROMs
!AutoROM --accept-license

import sys, gymnasium as gym, ale_py, torch, numpy as np, cv2, matplotlib, imageio, setuptools

print('=' * 65)
print(f'Python {sys.version.split()[0]} | Gymnasium {gym.__version__} | ALE {ale_py.__version__}')
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Verify ALE works
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5')
obs, _ = env.reset()
print(f'ALE/Pong-v5 obs shape: {obs.shape} | actions: {env.action_space}')
env.close()
print('=' * 65)
print('✅ Setup complete')

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/NatnaelTigistu/rl-atari-evolution.git'
REPO_NAME = 'rl-atari-evolution'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    !git -C {REPO_NAME} pull

ROOT = os.path.abspath(REPO_NAME)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

print(f'📂 Working directory: {os.getcwd()}')

In [ ]:
import random
import glob
import torch
import numpy as np
import gymnasium as gym
from IPython.display import Video, display as ipy_display

from config import CFG
from src.wrappers import make_atari_env, make_eval_env
from visualization.display import init_virtual_display
from benchmarks.evaluator import plot_single_algorithm, plot_master_comparison

# Global config
SEED      = 42
EPISODES  = 500
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_STEPS = 20 * 30  # 20 s of gameplay at ~30 fps

# Seed everything
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Start virtual display (headless Xvfb)
_display = init_virtual_display()

# Verify wrapper chain
env = make_atari_env(CFG.ENV_NAME, seed=SEED)
obs_arr = np.asarray(env.reset(seed=SEED)[0])
assert obs_arr.shape == (4, 84, 84), f'Bad shape: {obs_arr.shape}'
assert obs_arr.dtype == np.float32
env.close()

print(f'Device : {DEVICE}')
print(f'Episodes per algo : {EPISODES}')
print(f'Obs shape  : {obs_arr.shape}  dtype={obs_arr.dtype}  range=[{obs_arr.min():.2f},{obs_arr.max():.2f}]')
print('✅ Globals and display ready')

In [ ]:
# Shared helper — record one episode and display inline
def record_and_show(agent, algo_name):
    video_dir = f'videos/{algo_name}'
    os.makedirs(video_dir, exist_ok=True)

    eval_env = make_eval_env(CFG.ENV_NAME, seed=0, render_mode='rgb_array')
    rec_env  = gym.wrappers.RecordVideo(
        eval_env,
        video_folder=video_dir,
        episode_trigger=lambda ep: ep == 0,
        name_prefix=f'{algo_name}_eval',
        video_length=MAX_STEPS,
    )

    obs, _ = rec_env.reset(seed=0)
    done, total_rew, step = False, 0.0, 0

    while not done and step < MAX_STEPS:
        obs_t = torch.tensor(
            np.asarray(obs, dtype=np.float32), dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            if hasattr(agent, 'policy'):                    # REINFORCE
                action, _ = agent.policy.get_action(obs_t)
            else:                                           # A2C / PPO
                action, _, _, _ = agent.network.get_action(obs_t)
        obs, reward, terminated, truncated, _ = rec_env.step(action)
        total_rew += float(reward)
        done = terminated or truncated
        step += 1

    rec_env.close()
    print(f'[{algo_name}] eval reward: {total_rew:.1f}  steps: {step}')

    videos = sorted(glob.glob(f'{video_dir}/*.mp4'))
    if videos:
        ipy_display(Video(videos[-1], embed=True, width=420))
    else:
        print('⚠️  No .mp4 found — ensure ffmpeg is installed.')
    return total_rew

print('record_and_show() helper defined ✓')

---
## Section 2 · REINFORCE (Baseline)

Pure Monte-Carlo policy gradient. No value function; high variance.

In [ ]:
!python -m src.train \
    --algo reinforce \
    --episodes {EPISODES} \
    --save-freq 50 \
    --log-interval 10 \
    --seed {SEED}

In [ ]:
import matplotlib.pyplot as plt

plot_single_algorithm(
    csv_path        = 'logs/reinforce_rewards.csv',
    title           = 'REINFORCE Baseline',
    output_png_path = 'logs/reinforce_curve.png',
)
plt.show()

In [ ]:
from src.reinforce import REINFORCEAgent

ckpts = sorted(
    glob.glob('checkpoints/reinforce/ep_*.pt'),
    key=lambda p: int(os.path.basename(p).replace('ep_','').replace('.pt',''))
) or glob.glob('checkpoints/reinforce/final.pt')

if ckpts:
    agent_rf = REINFORCEAgent(action_dim=6, device=DEVICE)
    agent_rf.load(ckpts[-1])
    agent_rf.policy.eval()
    rf_eval_reward = record_and_show(agent_rf, 'reinforce')
else:
    print('⚠️  No REINFORCE checkpoint — run training cell first.')

---
## Section 3 · Vanilla A2C (N-Step Bootstrap)

Actor-Critic with shared CNN, N-step returns. Lower variance than REINFORCE.

In [ ]:
!python -m src.train \
    --algo a2c \
    --episodes {EPISODES} \
    --save-freq 50 \
    --log-interval 10 \
    --seed {SEED}

In [ ]:
plot_single_algorithm(
    csv_path        = 'logs/a2c_rewards.csv',
    title           = 'Vanilla A2C (N-Step Bootstrap)',
    output_png_path = 'logs/a2c_curve.png',
)
plt.show()

In [ ]:
from src.a2c import A2CAgent

ckpts = sorted(
    glob.glob('checkpoints/a2c/ep_*.pt'),
    key=lambda p: int(os.path.basename(p).replace('ep_','').replace('.pt',''))
) or glob.glob('checkpoints/a2c/final.pt')

if ckpts:
    agent_a2c = A2CAgent(action_dim=6, device=DEVICE)
    agent_a2c.load(ckpts[-1])
    agent_a2c.network.eval()
    a2c_eval_reward = record_and_show(agent_a2c, 'a2c')
else:
    print('⚠️  No A2C checkpoint — run training cell first.')

---
## Section 4 · A2C + GAE (Generalized Advantage Estimation)

GAE-λ smoothly interpolates between 1-step TD (λ=0) and Monte-Carlo (λ=1).

In [ ]:
!python -m src.train \
    --algo a2c_gae \
    --episodes {EPISODES} \
    --save-freq 50 \
    --log-interval 10 \
    --seed {SEED}

In [ ]:
plot_single_algorithm(
    csv_path        = 'logs/a2c_gae_rewards.csv',
    title           = 'A2C with GAE',
    output_png_path = 'logs/a2c_gae_curve.png',
)
plt.show()

In [ ]:
from src.a2c_gae import A2CGAEAgent

ckpts = sorted(
    glob.glob('checkpoints/a2c_gae/ep_*.pt'),
    key=lambda p: int(os.path.basename(p).replace('ep_','').replace('.pt',''))
) or glob.glob('checkpoints/a2c_gae/final.pt')

if ckpts:
    agent_gae = A2CGAEAgent(action_dim=6, device=DEVICE)
    agent_gae.load(ckpts[-1])
    agent_gae.network.eval()
    gae_eval_reward = record_and_show(agent_gae, 'a2c_gae')
else:
    print('⚠️  No A2C+GAE checkpoint — run training cell first.')

---
## Section 5 · PPO (Proximal Policy Optimization)

Clipped surrogate objective + GAE + K-epoch mini-batch updates. State of the art.

In [ ]:
!python -m src.train \
    --algo ppo \
    --episodes {EPISODES} \
    --save-freq 50 \
    --log-interval 10 \
    --seed {SEED}

In [ ]:
plot_single_algorithm(
    csv_path        = 'logs/ppo_rewards.csv',
    title           = 'PPO with Clipped Surrogate',
    output_png_path = 'logs/ppo_curve.png',
)
plt.show()

In [ ]:
from src.ppo import PPOAgent

ckpts = sorted(
    glob.glob('checkpoints/ppo/ep_*.pt'),
    key=lambda p: int(os.path.basename(p).replace('ep_','').replace('.pt',''))
) or glob.glob('checkpoints/ppo/final.pt')

if ckpts:
    agent_ppo = PPOAgent(action_dim=6, device=DEVICE)
    agent_ppo.load(ckpts[-1])
    agent_ppo.network.eval()
    ppo_eval_reward = record_and_show(agent_ppo, 'ppo')
else:
    print('⚠️  No PPO checkpoint — run training cell first.')

---
## Section 6 · Master Evolution Benchmark & Comparison

In [ ]:
import os
from IPython.display import Image, display as ipy_display

# Map display names → actual CSV paths written by train.py
LOG_MAP = {
    'REINFORCE':   'logs/reinforce_rewards.csv',
    'Vanilla A2C': 'logs/a2c_rewards.csv',
    'A2C + GAE':   'logs/a2c_gae_rewards.csv',
    'PPO':         'logs/ppo_rewards.csv',
}

out_png = plot_master_comparison(
    log_dict        = LOG_MAP,
    output_png_path = 'benchmarks/plots/evolution_comparison.png',
)

if os.path.exists(out_png):
    ipy_display(Image(filename=str(out_png), width=900))

In [ ]:
import pandas as pd

WINDOW = 100
rows = []

for algo, csv_path in LOG_MAP.items():
    if not os.path.exists(csv_path):
        rows.append({'Algorithm': algo, 'Final 100-ep avg': 'N/A',
                     'Max score': 'N/A', 'Training time (s)': 'N/A'})
        continue
    df = pd.read_csv(csv_path)
    final_avg = df['reward'].tail(WINDOW).mean()
    max_score = df['reward'].max()
    total_time = df['elapsed_s'].iloc[-1] if 'elapsed_s' in df.columns else float('nan')
    rows.append({
        'Algorithm':          algo,
        f'Final {WINDOW}-ep avg': f'{final_avg:.2f}',
        'Max score':          f'{max_score:.1f}',
        'Training time (s)':  f'{total_time:.0f}',
    })

summary = pd.DataFrame(rows)
print('\n' + '='*60)
print('  ALGORITHM PERFORMANCE SUMMARY')
print('='*60)
print(summary.to_string(index=False))
print('='*60)